# 20 — LeukemiPrediag v2.7 Measurement Robustness

**Research use only. NO leukemia outcome predictions. Frozen point forecasting.**

This notebook executes the versioned `src/feature_robustness_v27.py` pipeline and `src/finalize_v27.py` on pre-existing v2.3/v2.5 tables. Neither `clone_inference.py` nor `progression_features.py` is modified.

### Data protections
- V0: historical >=1% filtered dataset (source selection remains a limitation).
- V1: the full table retains exact LoD-floored values in an **audit-only** column; ordinary growth/clone fitting is allowed only on >=3 **quantifiable** timepoints (row LoD known, above limit; depth >=500 for LBC).
- V2: V1 plus strictly pre-index 1% detection, calculated before the last **patient** wave.
- Source-wide 1% ascertainment remains present; absence of a variant in this dataset is NOT a negative read.
- Two time sensitivity runs: original and approximate cohort-wave ages for GSE288742 (not individual collection times).
- Feature status: the pre-specified rho >=.8 in every cohort with >=20 paired patients, at least 2 such cohorts.
- When 80th-percentile ties make the top quintile non-identifiable, report NA.
- No clinical outcome label is used or created.

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    MYDRIVE=Path('/content/drive/MyDrive')
    candidates=[MYDRIVE/'leukemia-prediagnose'/'blood-cancer-early-detection',MYDRIVE/'blood-cancer-early-detection']
    candidates.extend(MYDRIVE.glob('*/blood-cancer-early-detection'))
except ImportError:
    candidates=[Path.cwd(),Path.cwd().parent]
ROOT=next((p for p in candidates if (p/'src'/'feature_robustness_v27.py').exists()),None)
assert ROOT is not None, f'Cannot find versioned v2.7 sources: {candidates}'
os.chdir(ROOT)
print('ROOT',ROOT)
V2=ROOT/'data'/'processed'/'v2'
for f in [V2/'real_serial_filtered.csv',V2/'real_serial_unfiltered.csv']:
    assert f.exists(),str(f)
OUT=V2/'feature_robustness_v27'
OUT.mkdir(exist_ok=True,parents=True)
print('OUT',OUT)

In [ ]:
import subprocess
# VAF point forecasting is frozen; these run ONLY the label-free measurement audit.
cmd=[sys.executable,'src/feature_robustness_v27.py','--filtered',str(V2/'real_serial_filtered.csv'),
     '--unfiltered',str(V2/'real_serial_unfiltered.csv'), '--outdir',str(OUT),'--draws','500','--seed','17']
p=subprocess.run(cmd,text=True,capture_output=True)
print(p.stdout[-2000:]);print(p.stderr[-2000:])
p.check_returncode()
subprocess.run([sys.executable,'src/finalize_v27.py',str(OUT)],check=True)
subprocess.run([sys.executable,'-m','unittest','-q','src/test_feature_robustness_v27.py'],check=True,
               env={**os.environ,'PYTHONPATH':str(ROOT/'src')})

In [ ]:
import json,pandas as pd
manifest=json.loads((OUT/'v27_manifest.json').read_text())
for axis in ['original','approx_age']:
    print('TIME',axis,manifest['runs'][axis]['sizes'])
    display(pd.read_csv(OUT/f'{axis}_feature_status.csv'))
    comp=pd.read_csv(OUT/f'{axis}_spearman_and_quintile.csv')
    display(comp[(comp.left=='V0')&(comp.right=='V2_PREINDEX_RELIABLE')].sort_values(['feature','cohort']))
    display(pd.read_csv(OUT/f'{axis}_within_patient.csv'))
    display(pd.read_csv(OUT/f'{axis}_spearman_bootstrap_ci.csv'))

## Scientific decision

`confirmatory_v27` means **passes the internal, frozen rank-robustness criterion only**. It is NOT clinical confirmation, prognostic validation, or proof of causal clone biology.

Any rank instability, missing/undefined ratio, poor time stability, assay-specific limitation, or unpowered cohort stays visible. Do not tune thresholds to promote features. Detailed limitations and a publication/clinical-readiness checklist are in `V27_MODEL_AUDIT.md` and `RESEARCH_TO_CLINICAL_VALIDATION_PLAN.md`.